# 개인 통합 정리용 공통 템플릿 작성본

- 프로젝트명: 성남시 젠트리피케이션 예측/분석
- 담당자명: 지륜
- 담당 파트: 통신 T데이터 통합/전처리/EDA, 공시지가 상승률 EDA
- 작성일: 2026-04-29
- 이 노트북의 목적: 내가 맡은 통신 T데이터와 공시지가 EDA의 입력 파일, 전처리 기준, 최종 산출물, 변수 후보를 공통 양식에 맞춰 정리한다.
- 최종 산출물 한 줄 요약: `T4~T27` 통신 테이블과 공시지가 2023~2025 상승률 EDA를 함께 정리했다.

> 이 파일은 공통 템플릿 형식에 맞춘 요약본이다. 상세한 테이블별 매핑과 EDA 판단은 `개인_통합정리_지륜템플릿_작성본.ipynb`에서 확인한다.


## 1. 작업 개요

| 항목 | 내용 |
|---|---|
| 내가 맡은 데이터/업무 | 통신 T데이터 `T4~T27` 통합 산출물 정리, 전처리 기준 문서화, EDA 기반 변수 후보 정리, 공시지가 상승률 EDA 정리 |
| 왜 필요한지 | 팀 최종 분석에서 통신 데이터를 행정동-월 또는 행정동-분기 단위 feature로 결합하기 위해 필요 |
| 최종적으로 남길 것 | 최종 parquet/csv 파일 목록, 처리 기준, 문제 데이터 표, 변수 연결표, 통계 검증 상태 |
| 현재 완료 범위 | 파일 존재/기간/행수/컬럼/핵심 결측 검증, 전처리 기준 정리, EDA 기반 후보 변수 정리 |
| 아직 남은 범위 | 최종 목표변수와 마스터 테이블 확정 후 통계적 유의성 및 모델링 검증 |

### 정리 범위
- 포함: `T4, T5, T6, T7, T8, T9, T10, T11, T12, T13, T14, T16, T20, T21, T22, T23, T24, T25, T26, T27`
- 현재 폴더 기준 미확보: `T15, T17, T18, T19`
- 추가 포함 EDA: `공시지가_test.ipynb`, `data/2차전처리_공시지가데이터.csv`


## 2. 입력 파일 정리

| 구분 | 파일/노트북 | 설명 | 사용 여부 |
|---|---|---|---|
| 최종 T데이터 | `data/t*_2023_2025_all_*` | T4~T27 최종 parquet/csv 산출물 | 사용 |
| 전처리 기록 | `Jiryun/1차_전처리_small.ipynb` | T4~T12, T14, T16, T20~T24 기초 전처리 | 근거 |
| 전처리 기록 | `Jiryun/1차_전처리.ipynb` | T13, T25, T26, T27 상세 전처리 | 근거 |
| EDA 기록 | `Jiryun/eda.ipynb` | 월별 유동/외부유입/인구 결합 EDA | 근거 |
| EDA 기록 | `Jiryun/my_eda.ipynb` | 분기별 EDA 및 변수 후보 정리 | 근거 |
| EDA 백업 | `Jiryun/my_eda_월별EDA_백업.ipynb` | 월별 행정동 단위 통합 EDA | 근거 |
| 공시지가 원본 | `data/2차전처리_공시지가데이터.csv` | 2023~2025 공시지가 상승률 EDA 입력 파일 | 사용 |
| 공시지가 EDA | `Jiryun/공시지가_test.ipynb` | 필지별/법정동별/구별 공시지가 상승률 계산 및 시각화 | 사용 |


In [ ]:
# [환경 설정] 기본 라이브러리와 경로
from pathlib import Path
import pandas as pd
import numpy as np
import pyarrow.parquet as pq
from IPython.display import display

pd.set_option('display.max_columns', 200)
pd.set_option('display.width', 200)
pd.set_option('display.max_rows', 100)

PERSON_NAME = '지륜'
PERSON_PART = '통신 T데이터 통합/전처리/EDA'

NOTEBOOK_DIR = Path.cwd()
PROJECT_ROOT = NOTEBOOK_DIR if (NOTEBOOK_DIR / 'data').exists() else NOTEBOOK_DIR.parent
DATA_DIR = PROJECT_ROOT / 'data'
JIRYUN_DIR = PROJECT_ROOT / 'Jiryun'

print(f'PROJECT_ROOT: {PROJECT_ROOT.resolve()}')
print(f'DATA_DIR    : {DATA_DIR.resolve()}')
print(f'JIRYUN_DIR  : {JIRYUN_DIR.resolve()}')


## 3. 파일 존재 여부 확인

최종 산출물 후보 파일만 먼저 확인한다. 대용량 parquet는 전체를 읽지 않고 메타데이터로 행 수와 컬럼 수를 확인한다.


In [ ]:
source_files = {
    'T4':  DATA_DIR / 't4_2023_2025_all_date_final.parquet',
    'T5':  DATA_DIR / 't5_2023_2025_all_date_final.parquet',
    'T6':  DATA_DIR / 't6_2023_2025_all_date_final.parquet',
    'T7':  DATA_DIR / 't7_2023_2025_all_date_final.parquet',
    'T8':  DATA_DIR / 't8_2023_2025_all_date_final.parquet',
    'T9':  DATA_DIR / 't9_2023_2025_all_date_final.parquet',
    'T10': DATA_DIR / 't10_2023_2025_all_date_final.parquet',
    'T11': DATA_DIR / 't11_2023_2025_all_date_final.parquet',
    'T12': DATA_DIR / 't12_2023_2025_all_final_v2.parquet',
    'T13': DATA_DIR / 't13_2023_2025_all_final_v2.parquet',
    'T14': DATA_DIR / 't14_2023_2025_all_final_v2.parquet',
    'T16': DATA_DIR / 't16_2023_2025_all_date_final.parquet',
    'T20': DATA_DIR / 't20_2023_2025_all_date_final.csv',
    'T21': DATA_DIR / 't21_2023_2025_all_date_final.csv',
    'T22': DATA_DIR / 't22_2023_2025_all_date_final.parquet',
    'T23': DATA_DIR / 't23_2023_2025_all_date_final.parquet',
    'T24': DATA_DIR / 't24_2023_2025_all_date_final.parquet',
    'T25': DATA_DIR / 't25_2023_2025_all_final_v2.parquet',
    'T26': DATA_DIR / 't26_2023_2025_all_final.parquet',
    'T27': DATA_DIR / 't27_2023_2025_all_final.parquet',
    '공시지가': DATA_DIR / '2차전처리_공시지가데이터.csv',
}

rows = []
for table, path in source_files.items():
    row = {'table': table, 'file': path.name, 'exists': path.exists(), 'size_mb': np.nan, 'rows': np.nan, 'columns': np.nan}
    if path.exists():
        row['size_mb'] = round(path.stat().st_size / 1024 / 1024, 2)
        if path.suffix.lower() == '.parquet':
            pf = pq.ParquetFile(path)
            row['rows'] = pf.metadata.num_rows
            row['columns'] = len(pf.schema_arrow.names)
        elif path.suffix.lower() == '.csv':
            row['rows'] = sum(1 for _ in path.open('rb')) - 1
            row['columns'] = len(pd.read_csv(path, nrows=0).columns)
    rows.append(row)

file_check = pd.DataFrame(rows)
file_check['rows'] = file_check['rows'].astype('Int64')
file_check['columns'] = file_check['columns'].astype('Int64')
display(file_check)

missing = file_check.loc[~file_check['exists'], ['table', 'file']]
if len(missing):
    print('[확인 필요] 없는 파일이 있습니다.')
    display(missing)
else:
    print('최종 후보 파일은 모두 존재합니다.')


## 4. 데이터 로드

대용량 파일이 많으므로 공통 작성본에서는 전체 데이터를 한 번에 로드하지 않는다. 필요할 때만 개별 파일을 샘플 또는 DuckDB/parquet 메타데이터로 확인한다.


In [ ]:
def load_table(path: Path, **kwargs):
    suffix = path.suffix.lower()
    if suffix == '.csv':
        return pd.read_csv(path, **kwargs)
    if suffix in {'.xlsx', '.xls'}:
        return pd.read_excel(path, **kwargs)
    if suffix == '.parquet':
        return pd.read_parquet(path, **kwargs)
    raise ValueError(f'지원하지 않는 파일 형식입니다: {suffix}')


def clean_columns(df: pd.DataFrame) -> pd.DataFrame:
    result = df.copy()
    result.columns = [str(col).strip() for col in result.columns]
    return result

# 예시: 작은 CSV나 필요한 parquet만 골라서 확인
# sample_df = clean_columns(load_table(source_files['T20']))
# display(sample_df.head())


## 5. 데이터 기본 점검

현재 완료된 기본 점검은 다음과 같다.

| 점검 항목 | 상태 | 설명 |
|---|---|---|
| 행 수/컬럼 수 | 완료 | parquet 메타데이터와 CSV row count로 확인 |
| 기간 범위 | 완료 | `ETL_YM`, `ETL_YMD` 기준 2023-01~2025-12 확인 |
| 핵심 컬럼 결측 | 완료 | `CNT`, `DURATION`, `PURPOSE`, `TRANS_GB`, `SEX_CD`, `AGE_GRP` 중심 확인 |
| 코드값 분포 | 일부 완료 | 기존 전처리/EDA 노트북에서 확인 |
| 통계적 유의성 | 미완료 | 최종 feature table과 목표변수 확정 후 검정 필요 |


In [ ]:
WATCH_COLUMNS = ['CNT', 'DURATION', 'PURPOSE', 'TRANS_GB', 'SEX_CD', 'AGE_GRP', 'D_CTY_NM', 'D_ADMI_NM', 'O_CTY_NM', 'O_ADMI_NM', 'CTY_NM', 'ADMI_NM']

def parquet_null_count(pf: pq.ParquetFile, col: str):
    cols = pf.schema_arrow.names
    if col not in cols:
        return None
    idx = cols.index(col)
    total = 0
    has_stats = False
    for row_group_idx in range(pf.metadata.num_row_groups):
        stats = pf.metadata.row_group(row_group_idx).column(idx).statistics
        if stats is not None:
            total += stats.null_count or 0
            has_stats = True
    return total if has_stats else np.nan

null_rows = []
for table, path in source_files.items():
    if not path.exists():
        continue
    if path.suffix.lower() == '.parquet':
        pf = pq.ParquetFile(path)
        cols = pf.schema_arrow.names
        for col in WATCH_COLUMNS:
            if col in cols:
                null_rows.append({'table': table, 'column': col, 'null_count': parquet_null_count(pf, col)})
    elif path.suffix.lower() == '.csv':
        df = pd.read_csv(path)
        for col in WATCH_COLUMNS:
            if col in df.columns:
                null_rows.append({'table': table, 'column': col, 'null_count': int(df[col].isna().sum())})

null_df = pd.DataFrame(null_rows)
if not null_df.empty:
    display(null_df.pivot(index='table', columns='column', values='null_count').fillna(''))


## 6. 문제 데이터 정리 섹션

| 데이터셋 | 컬럼명/영역 | 문제 유형 | 처리 방식 | 이유 |
|---|---|---|---|---|
| T12 | `D_CTY_NM` | 일부 지역명 결측 | 기록 후 코드/좌표 우선 확인 | 핵심 CNT/코드 컬럼은 결측 없음 |
| T13 | O/D 지역명 | 소수 지역명 결측 및 99 코드 | 보정 가능한 값만 보정, 99는 유지 | 공급 데이터의 미확인 의미 보존 |
| T14 | O/D 시군구명 | 일부 지역명 결측 | 기록 후 필요 시 코드 기준 보조 | 핵심 교통/성연령/CNT는 결측 없음 |
| T25 | O/D 시군구명 | 소수 지역명 결측 및 99 코드 | 보정 가능한 값만 보정, 99는 유지 | 임의 삭제 시 이동량 왜곡 가능 |
| T21 | `CNT` 없음 | 유동량 직접 변수 없음 | 참고용으로 유지 | 모델 feature 후보에서 제외 |
| T15/T17/T18/T19 | 파일 없음 | 현재 폴더 기준 미확보 | 누락 테이블로 표시 | 팀 공유 시 범위 혼선 방지 |
| 공시지가 | 법정동/행정동 단위 | 통신 데이터와 공간 단위가 다름 | 결합 전 매핑 기준 별도 확인 | 법정동 상승률을 행정동 feature로 바로 단정하면 안 됨 |
| 공시지가 | 표본 수 작은 법정동 | 순위 해석 불안정 | `필지수 >= 20` 조건 사용 | 중앙값 안정성 확보 |


## 7. 처리 기준 문서화

- 원본/중간 파일은 직접 수정하지 않고, 분석용 최종 산출물을 별도로 관리한다.
- parquet 저장 가능 테이블은 parquet를 최종 후보로 사용한다.
- T20/T21처럼 기존 최종 산출물이 CSV인 경우 CSV로 유지한다.
- 날짜는 `ETL_YM` 또는 `ETL_YMD` 기준으로 2023~2025 범위를 확인한다.
- 지역명 결측은 코드 기준으로 보정 가능한 경우만 보정하고, `99` 미확인 코드는 원본 의미 보존을 위해 유지한다.
- `PURPOSE`, `TRANS_GB`는 코드 정의를 반영하되 통신 추정값이므로 실제 목적/수단으로 단정하지 않는다.
- 현재 변수 후보는 통계적 유의성이 확정된 변수가 아니라 EDA 기반 후보이다.
- 공시지가는 2023/2024/2025년 1월 기준 같은 `고유번호` 필지의 상승률을 계산하고, 법정동 대표값은 평균보다 중앙값을 우선 사용한다.


## 8. 최종 산출물 저장

이미 전처리 노트북에서 최종 parquet/csv 파일이 저장되어 있으므로, 이 공통 작성본에서는 새 파일을 다시 저장하지 않는다. 대신 최종 산출물 존재 여부와 재오픈 가능 여부를 검증한다.


In [ ]:
reopen_rows = []
for table, path in source_files.items():
    if not path.exists():
        reopen_rows.append({'table': table, 'file': path.name, 'reopen_ok': False, 'note': 'file missing'})
        continue
    try:
        if path.suffix.lower() == '.parquet':
            pf = pq.ParquetFile(path)
            note = f'rows={pf.metadata.num_rows}, cols={len(pf.schema_arrow.names)}'
        else:
            sample = pd.read_csv(path, nrows=5)
            note = f'sample_rows={len(sample)}, cols={len(sample.columns)}'
        reopen_rows.append({'table': table, 'file': path.name, 'reopen_ok': True, 'note': note})
    except Exception as exc:
        reopen_rows.append({'table': table, 'file': path.name, 'reopen_ok': False, 'note': str(exc)})

reopen_check = pd.DataFrame(reopen_rows)
display(reopen_check)


## 8-1. 공시지가 EDA 요약

| 항목 | 내용 |
|---|---|
| 입력 파일 | `data/2차전처리_공시지가데이터.csv` |
| 작업 노트북 | `Jiryun/공시지가_test.ipynb` |
| 데이터 규모 | 15,542행, 13컬럼 |
| 기간 | 2023~2025년, 1월 기준 중심. 7월 자료는 소수 존재 |
| 공간 단위 | 법정동/구, 필지 고유번호 기준 |
| 주요 컬럼 | `고유번호`, `법정동코드`, `법정동명`, `구`, `기준연도`, `기준월`, `공시지가` |
| 계산 방식 | 같은 `고유번호` 필지의 2023/2024/2025년 1월 공시지가를 맞춘 뒤 상승률 계산 |
| 파생 지표 | `상승률_23_24`, `상승률_24_25`, `상승률_23_25`, `연평균상승률_23_25` |
| 집계 기준 | 법정동별/구별 중앙값 중심, 법정동 표본은 `필지수 >= 20` 조건 적용 |
| 해석 주의 | 법정동 기준이므로 통신 행정동 feature와 결합할 때 매핑 기준 확인 필요 |

### 공시지가 EDA 주요 결과
- 1월 기준 전체 고유번호 5,178개 중 2023~2025가 모두 존재하는 필지는 5,176개다.
- 법정동은 43개이며, 표본 수 20개 이상 조건을 적용하면 40개 법정동이 비교 대상이다.
- 2023~2025 누적 상승률 중앙값 상위는 시흥동, 삼평동, 백현동, 금토동, 서현동 순으로 확인했다.
- 하위는 상적동, 상대원동, 금광동, 여수동, 도촌동 순으로 확인했다.
- 상승률이 높다는 것은 상승 속도가 빠르다는 뜻이지, 공시지가 수준 자체가 높다는 뜻은 아니므로 가격 수준과 상승률을 분리해 해석한다.


## 8-2. 보조 작업 간단 메모

아래 내용은 주요 결론이 아니라 작업 중 확인한 보조 메모다.

| 항목 | 간단 메모 |
|---|---|
| T20 | `DISTANCE`, `CARBON_EMISSIONS`가 있어 거리/탄소 참고 변수로만 기록 |
| T21 | `CNT`가 없어 직접 유동량 feature로 쓰기보다 시간대 참고용으로 기록 |
| T11 | 작업 중 중복 확인 과정이 있었지만 최종 파일 기준으로 정리됨. 중요 이슈로 따로 해석하지 않음 |
| T22 | 생산가능/소비가능 연령층 비중은 보조 후보로만 기록 |
| 통계 점검 | 기존 EDA에서 일부 확인했지만 전체 feature 검증은 최종 모델링 단계에서 진행 |


## 9. 최종 산출물 요약표

| 원본/중간 범위 | 최종 파일 | 기간 범위 | 핵심 컬럼 | 최종 사용 목적 |
|---|---|---|---|---|
| T4~T11 | `t4~t11_2023_2025_all_date_final.parquet` | 2023-01~2025-12 | `CNT`, `PURPOSE` 또는 `TRANS_GB`, `SEX_CD`, `AGE_GRP` | 구조 파악/보조 feature |
| T12/T14 | `t12/t14_2023_2025_all_final_v2.parquet` | 2023-01~2025-12 | O/D 시군구, `PURPOSE` 또는 `TRANS_GB`, `CNT` | 시군구 OD 보조 |
| T13 | `t13_2023_2025_all_final_v2.parquet` | 2023-01-01~2025-12-31 | O/D 행정동, `PURPOSE`, `CNT` | 외부유입/유출입 핵심 후보 |
| T16 | `t16_2023_2025_all_date_final.parquet` | 2023-01~2025-12 | `PURPOSE`, `DURATION`, `CNT` | 체류시간 보조 |
| T20/T21 | `t20/t21_2023_2025_all_date_final.csv` | 2023-01~2025-12 | `TRANS_GB`, `TIME_CD` 등 | 참고용 |
| T22~T24 | `t22~t24_2023_2025_all_date_final.parquet` | 2023-01-01~2025-12-31 | 행정동, 시간대, 성/연령, 목적, `CNT` | 인구 구성/유동인구 후보 |
| T25~T27 | `t25_final_v2`, `t26_final`, `t27_final` | 2023-01-01~2025-12-31 | O/D, 목적, 이동수단, 체류시간, `CNT` | 이동/체류/접근성 후보 |
| 공시지가 | `2차전처리_공시지가데이터.csv`, `공시지가_test.ipynb` | 2023~2025 | `공시지가`, `상승률_23_25`, `연평균상승률_23_25` | 지역 토지가치 변화 보조지표 |

## 10. 최종 변수 연결표

| 데이터셋 | 원본 컬럼 | 최종 변수명/후보 | 의미 | 사용 방향 |
|---|---|---|---|---|
| T24 | `CNT` | `q_floating_pop_sum` | 행정동 분기 유동량 | 기본 규모 변수 |
| T24 | `TIME_CD` | `q_night_ratio`, `q_lunch_ratio`, `q_evening_ratio` | 시간대별 비중 | 상권 시간대 특성 |
| T24 | `PURPOSE` | `q_purpose_*_ratio` | 목적별 유동 비중 | 지역 성격 비교 |
| T13 | O/D 행정동, `CNT` | `q_external_inflow_cnt`, `q_external_inflow_ratio` | 외부유입 규모/비중 | 외부 수요 proxy |
| T26 | `DURATION`, `CNT` | `q_avg_stay_time`, `q_long_stay_ratio` | 체류시간/장기체류 비중 | 체류형 상권 여부 |
| T27 | `TRANS_GB`, `CNT` | `q_transport_*_ratio` | 이동수단별 비중 | 접근성/교통 특성 |
| T22 | 성/연령별 CNT 컬럼 | `q_age_sex_foreigner_mix` | 이용자 구성 | 보조 변수 |
| T22 | 성/연령별 CNT 컬럼 | `working_age_share`, `consumer_age_share` | 생산가능/소비가능 연령층 비중 | 인구 구성 보조 |
| T20 | `DISTANCE`, `CARBON_EMISSIONS` | `distance_sum`, `carbon_emissions_sum` | 이동 거리/탄소배출량 | 참고용 보조 변수 |
| 공시지가 | `공시지가` | `official_land_price_2023/2024/2025` | 연도별 필지 공시지가 | 가격 수준 보조 |
| 공시지가 | `고유번호`, `기준연도`, `공시지가` | `official_land_price_growth_23_25` | 2023~2025 누적 상승률 | 토지가치 상승 속도 |
| 공시지가 | `법정동명`, 상승률 | `dong_land_price_growth_median` | 법정동별 상승률 중앙값 | 지역 단위 보조 feature |


## 11. 최종 체크리스트

- [x] 원본/중간 파일명 정리 완료
- [x] 파일 존재 여부 확인 완료
- [x] 데이터 기본 점검 완료
- [x] 문제 데이터 정리표 작성 완료
- [x] 처리 기준 문서화 완료
- [x] 전처리 코드/노트북 역할 정리 완료
- [x] clean/final 파일 저장 결과 확인 완료
- [x] 최종 산출물 요약표 작성 완료
- [x] 최종 변수 연결표 작성 완료
- [x] 통계 검증 완료/미완료 상태 구분 완료
- [x] 공시지가 EDA 정리 포함 완료
- [x] 기존 노트북 재대조 후 세부 누락 보완 완료

## 12. 마지막 요약

1. 통신 T데이터는 현재 확보된 `T4~T27` 최종 산출물 기준으로 정리했고, 공시지가 상승률 EDA도 함께 포함했다.
2. `T15`, `T17`, `T18`, `T19`는 현재 폴더 기준 파일이 없어 누락 테이블로 표시했다.
3. 파일 존재, 행 수, 컬럼 수, 기간 범위, 핵심 컬럼 결측 검증은 완료했다.
4. `T24 + T13 + T26`은 EDA 기반 1차 후보이며, 통계적 유의성이 확정된 것은 아니다.
5. 최종 목표변수와 마스터 테이블 확정 후 통계 검정, 모델링 검증, 법정동-행정동 매핑 검증을 추가해야 한다.
